<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/02_LiLt_label_projection_and_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate seqeval huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import json
import pandas as pd
from pathlib import Path
from huggingface_hub import snapshot_download

In [ ]:
REPO_ID = "Satyamkarn100/FinDoc-AI"
WORK_DIR = Path("./findoc_work").resolve()
HF_DIR = WORK_DIR / "hf_data"
ANN_DIR = WORK_DIR / "fatura_ann"
WORK_DIR.mkdir(parents=True,exist_ok=True)

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=HF_DIR,
    allow_patterns=["manifests/*", "ocr/doctr_default/*.jsonl"],
)

manifest = pd.read_csv(HF_DIR / "manifests/manifest.csv")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
ocr_data = {}
for shard in sorted((HF_DIR / "ocr").rglob("shard_*.jsonl")):
    print("Loading:", shard.name)
    with open(shard, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            ocr_data[rec["filename"]] = rec

print("Total OCR loaded:", len(ocr_data))

Loading: shard_0000.jsonl
Loading: shard_0001.jsonl
Loading: shard_0002.jsonl
Total OCR loaded: 10000


In [ ]:
ORIG_DIR = ANN_DIR / "invoices_dataset_final/Annotations/Original_Format"
if not ORIG_DIR.exists():
    zip_path = WORK_DIR / "FATURA2.zip"
    !wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O "{zip_path}"
    !unzip -q -o "{zip_path}" "invoices_dataset_final/Annotations/Original_Format/*" -d "{ANN_DIR}"

print("Manifest rows :", len(manifest))
print("OCR loaded    :", len(ocr_data))
print("JSONs loaded  :", len(list(ORIG_DIR.glob("*.json"))))
print("\nSplit counts:\n", manifest["split"].value_counts())

--2026-11-26 18:10:17--  https://zenodo.org/records/10371464/files/FATURA2.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.45.63, 137.138.121.78, 188.185.47.192, ...
Connecting to zenodo.org (zenodo.org)|188.185.45.63|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 690717658 (659M) [application/octet-stream]
Saving to: ‘/kaggle/working/findoc_work/FATURA2.zip’

/kaggle/working/fin 100%[===================>] 658.72M  5.53MB/s    in 2m 49s  

2026-11-26 18:13:06 (3.91 MB/s) - ‘/kaggle/working/findoc_work/FATURA2.zip’ saved [690717658/690717658]

Manifest rows : 10000
OCR loaded    : 10000
JSONs loaded  : 10000

Split counts:
 split
train    6400
test     2000
val      1600
Name: count, dtype: int64


In [ ]:
file_path = ORIG_DIR / "Template1_Instance0.json"

with open(file_path,'r') as f:
    ann = json.load(f)

print(ann["TITLE"])

{'bbox': [[328.0, 802.1638], [443.02, 820.1638]], 'text': 'TAX INVOICE'}


In [ ]:
bbox = ann["TITLE"]["bbox"]
x1,y1 = bbox[0]
x2,y2 = bbox[1]

rec = ocr_data['Template1_Instance0.jpg']
w = rec['width']
h = rec['height']


nx1 = min(x1,x2) / w
nx2 = max(x1,x2) / w

ny1 = (h-max(y1,y2)) / h
ny2 = (h-min(y1,y2)) / h

data_box = [nx1,ny1,nx2,ny2]
print(data_box)

[0.5512605042016807, 0.024775505350772845, 0.7445714285714286, 0.04617859690844229]


In [ ]:
gx1, gy1, gx2, gy2 = data_box

for wd in rec["words"]:
    text = wd[0]
    wx1, wy1, wx2, wy2 = wd[2:6]

    ix1 = max(wx1, gx1)
    iy1 = max(wy1, gy1)
    ix2 = min(wx2, gx2)
    iy2 = min(wy2, gy2)

    if ix2 > ix1 and iy2 > iy1:
        inter_area = (ix2 - ix1) * (iy2 - iy1)
        word_area = (wx2 - wx1) * (wy2 - wy1)
        ratio = inter_area / word_area

        if ratio >= 0.5:
            print(text, round(ratio, 2))

TAX 0.97
INVOICE 1.0


In [ ]:
print(" ".join(wd[0] for wd in ocr_data["Template10_Instance0.jpg"]["words"]))

Address:4894 Melanie Pass Apt. 742 Tammyland, SD 42587 US Date: 03-Jan-1994 INVOICE # INV/30-14/832 Due Date : 06-Feb-2007 ITEMS Prove amount sell, Assume against. Terms and Conditions HGH Bill to:Michael Sparks 4894 Melanie Pass Apt. 742 Tammyland, SD 42587 US Tel:*/311)738-8211 Mclean-Cochran Email:cjames@example.com Site:https:/wwww.chase-allen.orgl QUANTITY 1.00 6.00 PRICE $64.54 $62.36 TAX $0.00 $0.00 DISCOUNT(2.14%): (-) 9.39 TOTAL : 441.14 EUR Email:cjames@example.com AMOUNT $64.54 $374.16 will be charged if payment is not made within the due date. Thank you for your business!


# Finalizing the Feilds

In [ ]:
FIELDS = (
    "NUMBER",
    "DATE",
    "DUE_DATE",
    "PO_NUMBER",
    "CUSTOMER",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "SELLER_EMAIL",
    "SUB_TOTAL",
    "DISCOUNT",
    "TAX",
    "TOTAL",
    "AMOUNT_DUE",
)

In [ ]:
labels = ["0"]
for field in FIELDS:
    labels.append(f"B-{field}")
    labels.append(f"I-{field}")

# Label Encoding
label2id = {}
id2label = {}

for idx,lbl in enumerate(labels):
    label2id[lbl] = idx
    id2label[idx] =lbl

print("Total fields:", len(FIELDS))
print("Total BIO classes:", len(labels))
print("Sample mapping:", list(label2id.items())[:7])


Total fields: 13
Total BIO classes: 27
Sample mapping: [('0', 0), ('B-NUMBER', 1), ('I-NUMBER', 2), ('B-DATE', 3), ('I-DATE', 4), ('B-DUE_DATE', 5), ('I-DUE_DATE', 6)]


In [ ]:
import json, re
from pathlib import Path
from collections import Counter, defaultdict

key_docs = Counter()
key_templates = defaultdict(set)
key_samples = defaultdict(dict)  # har key ke 3 alag templates ke samples

for p in sorted(Path(ORIG_DIR).glob("*.json")):
    m = re.match(r"Template(\d+)_", p.name)
    t = int(m.group(1)) if m else -1
    ann = json.load(open(p, encoding="utf-8"))

    for k, v in ann.items():
        key_docs[k] += 1
        key_templates[k].add(t)
        if len(key_samples[k]) < 3 and t not in key_samples[k]:
            if isinstance(v, dict):
                txt = str(v.get("text", "<no text>"))
            elif isinstance(v, list):
                txt = f"<list len={len(v)}>"
            else:
                txt = f"<{type(v).__name__}>"
            key_samples[k][t] = txt.replace("\n", " | ")[:75]

for k, n in key_docs.most_common():
    print(f"\n[{k}] -> {n} docs | {len(key_templates[k])} templates")
    for t, txt in key_samples[k].items():
        print(f"   T{t:02d}: {txt}")


[TABLE] -> 10000 docs | 50 templates
   T10: <list len=1>
   T11: <list len=1>
   T12: <list len=1>

[INVOICE_INFO] -> 10000 docs | 50 templates
   T10: <list len=0>
   T11: <list len=0>
   T12: <list len=0>

[OTHER] -> 10000 docs | 50 templates
   T10: Address:4894 Melanie Pass Apt. 742 | Tammyland, SD 42587 US |  | Date: 03-J
   T11: TAX INVOICE |  | Invoice number INV/72-06/714 | Invoice Date: 20-Apr-2010 |
   T12: INVOICE |  | Invoice number 2970-559 | Invoice Date: 23-Jan-2002 | Due Date

[LOGO] -> 10000 docs | 50 templates
   T10: <no text>
   T11: <no text>
   T12: <no text>

[DATE] -> 9800 docs | 49 templates
   T10: Date: 03-Jan-1994
   T11: Invoice Date: 20-Apr-2010
   T12: Invoice Date: 23-Jan-2002

[NUMBER] -> 8800 docs | 44 templates
   T10: INVOICE # INV/30-14/832
   T11: Invoice number INV/72-06/714
   T12: Invoice number 2970-559

[SELLER_ADDRESS] -> 8200 docs | 41 templates
   T10: Address:4894 Melanie Pass Apt. 742 | Tammyland, SD 42587 US
   T12: Address:6479 Smith 

In [ ]:
def cannon(ann):
    out = {}
    for k,v in ann.items():
        if k in ("BUYER","BILL_TO"):
            key = "CUSTOMER"

        elif k == "SEND_TO":
            key = "SHIP_TO"


        elif k == "TAX" or k == "GST" or k.startswith("GST("):
            key = "TAX"

        else:
            key = k

        if key not in out:
            out[key] = [v] # {"CUSTOMER": [Joyce_data]}
        else:
            out[key].append(v)

    return out



In [ ]:
sample = json.load(open(Path(ORIG_DIR) / "Template49_Instance167.json", encoding="utf-8"))
res = cannon(sample)
print("Keys:", list(res.keys()))
print("CUSTOMER count:", len(res.get("CUSTOMER", [])))

Keys: ['TABLE', 'INVOICE_INFO', 'CUSTOMER', 'DATE', 'NUMBER', 'SELLER_ADDRESS', 'SELLER_NAME', 'SUB_TOTAL', 'TOTAL', 'TOTAL_WORDS', 'GSTIN', 'OTHER', 'LOGO']
CUSTOMER count: 2


In [ ]:
print(res["CUSTOMER"][0])

{'bbox': [[22.0, 648.4058000000001], [215.07999999999998, 732.4058]], 'text': 'Bill to:Joyce Brock\n67052 Kelly Land\nNew Christyport, AR 92366 US\nTel:+(086)834-1218\nEmail:nstephenson@example.org\nSite:https://martinez-rivera.info/'}


In [ ]:
import numpy as np
def label_one_invoice(words, bboxes, canon_ann):
    if not words:
        return []

    labels = np.full(len(words), "O", dtype=object)
    skip_keys = {"OTHER", "LOGO"}

    # Ek hi baar saare word centers nikaal liye (chahe [x0,y0,x1,y1] ho ya [[x0,y0],[x1,y1]])
    arr = np.array(bboxes, dtype=float).reshape(-1, 4)
    cx = (arr[:, 0] + arr[:, 2]) / 2
    cy = (arr[:, 1] + arr[:, 3]) / 2

    for key, items in canon_ann.items():
        if key in skip_keys:
            continue

        for item in items:
            if not isinstance(item, dict) or "bbox" not in item:
                continue

            (ax0, ay0), (ax1, ay1) = item["bbox"]
            x_min, x_max = min(ax0, ax1), max(ax0, ax1)
            y_min, y_max = min(ay0, ay1), max(ay0, ay1)

            # 3rd loop ki jagah seedha NumPy filter
            idx = np.where((cx >= x_min) & (cx <= x_max) & (cy >= y_min) & (cy <= y_max))[0]
            if len(idx) > 0:
                labels[idx[0]] = f"B-{key}"
                labels[idx[1:]] = f"I-{key}"

    return labels.tolist()

In [ ]:
test_words = ["Bill", "to", "RandomOutsideWord"]
test_bboxes = [
    [30, 660, 80, 680],       # CUSTOMER box ke andar
    [85, 660, 140, 680],      # CUSTOMER box ke andar
    [5000, 5000, 5010, 5010]  # Bahar (O aana chahiye)
]

print(label_one_invoice(test_words, test_bboxes, res))

['B-CUSTOMER', 'I-CUSTOMER', 'O']


In [ ]:
canon_ann = cannon(raw_ann)

tagged = [(w, l) for w, l in zip(words, labels) if l != "O"]
print("Total words:", len(words), "| Tagged:", len(tagged))
print(tagged[:20])


NameError: name 'raw_ann' is not defined

In [ ]:
import json, pandas as pd
from pathlib import Path

root = Path("/kaggle/working/findoc_work")

df = pd.read_csv(root / "hf_data/manifests/manifest.csv")
print("Manifest shape:", df.shape)
print("Manifest columns:", df.columns.tolist())
print(df.head(2), "\n")
if "split" in df.columns:
    print("Splits:\n", df["split"].value_counts(), "\n")

for jl in sorted(root.rglob("*.jsonl")):
    with jl.open() as f:
        first_line = json.loads(f.readline())
    print("JSONL:", jl.relative_to(root))
    print("  Keys:", list(first_line.keys()))
    for k, v in first_line.items():
        preview = str(v)[:100]
        print(f"    {k}: {preview}")
    print()

ann_samples = sorted((root / "fatura_ann").rglob("*.json"))[:2]
print("Sample ann files:", [str(p.relative_to(root)) for p in ann_samples])

Manifest shape: (10000, 6)
Manifest columns: ['img_path', 'annot_path', 'template_id', 'instance_id', 'split', 'is_inlayout_holdout']
                   img_path                 annot_path  template_id  \
0  Template10_Instance0.jpg  Template10_Instance0.json           10   
1  Template10_Instance1.jpg  Template10_Instance1.json           10   

   instance_id  split  is_inlayout_holdout  
0            0  train                 True  
1            1  train                False   

Splits:
 split
train    6400
test     2000
val      1600
Name: count, dtype: int64 

JSONL: hf_data/ocr/doctr_default/shard_0000.jsonl
  Keys: ['filename', 'width', 'height', 'words']
    filename: Template10_Instance0.jpg
    width: 595
    height: 841
    words: [['Address:4894', 0.9815, 0.54279, 0.00195, 0.67806, 0.01562, 0], ['Melanie', 0.9997, 0.68082, 0.001

JSONL: hf_data/ocr/doctr_default/shard_0001.jsonl
  Keys: ['filename', 'width', 'height', 'words']
    filename: Template10_Instance112.jpg
    widt

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path("/kaggle/working/findoc_work")
ANN_DIR = ROOT / "fatura_ann/invoices_dataset_final/Annotations/Original_Format"
OCR_SHARDS = sorted((ROOT / "hf_data/ocr/doctr_default").glob("*.jsonl"))

# 1. Saare OCR shards load karo aur count dekho
ocr_index = {}
for shard in OCR_SHARDS:
    with shard.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            ocr_index[rec["filename"]] = rec

print(f"Total OCR records across {len(OCR_SHARDS)} shards: {len(ocr_index)}")

# 2. Manifest breakdown (split x is_inlayout_holdout)
df = pd.read_csv(ROOT / "hf_data/manifests/manifest.csv")
print("\nManifest breakdown:")
print(df.groupby(["split", "is_inlayout_holdout"]).size())

# 3. 1 real invoice par alignment sanity check
sample_img = "Template10_Instance0.jpg"
ocr_rec = ocr_index[sample_img]
W, H = ocr_rec["width"], ocr_rec["height"]

# OCR words -> 0-1000 int boxes (LiLT format)
words = [w[0] for w in ocr_rec["words"]]
bboxes_1000 = [
    [
        max(0, min(1000, int(round(w[2] * 1000)))),
        max(0, min(1000, int(round(w[3] * 1000)))),
        max(0, min(1000, int(round(w[4] * 1000)))),
        max(0, min(1000, int(round(w[5] * 1000)))),
    ]
    for w in ocr_rec["words"]
]

raw_ann = json.loads((ANN_DIR / sample_img.replace(".jpg", ".json")).read_text(encoding="utf-8"))
canon_ann = cannon(raw_ann)

def scale_ann_to_1000(c_ann, width, height, flip_y=True):
    out = {}
    for k, items in c_ann.items():
        new_items = []
        for it in items:
            if not isinstance(it, dict) or "bbox" not in it:
                continue
            (x0, y0), (x1, y1) = it["bbox"]
            nx0 = max(0, min(1000, int(round((min(x0, x1) / width) * 1000))))
            nx1 = max(0, min(1000, int(round((max(x0, x1) / width) * 1000))))
            if flip_y:
                ny0 = max(0, min(1000, int(round((1.0 - max(y0, y1) / height) * 1000))))
                ny1 = max(0, min(1000, int(round((1.0 - min(y0, y1) / height) * 1000))))
            else:
                ny0 = max(0, min(1000, int(round((min(y0, y1) / height) * 1000))))
                ny1 = max(0, min(1000, int(round((max(y0, y1) / height) * 1000))))
            new_items.append({**it, "bbox": [[nx0, ny0], [nx1, ny1]]})
        out[k] = new_items
    return out

for flip in [True, False]:
    norm_ann = scale_ann_to_1000(canon_ann, W, H, flip_y=flip)
    labels = label_one_invoice(words, bboxes_1000, norm_ann)
    tagged = [(w, l) for w, l in zip(words, labels) if l != "O"]
    print(f"\n--- flip_y={flip} | Tagged {len(tagged)} / {len(words)} words ---")
    print(tagged[:20])

Total OCR records across 3 shards: 10000

Manifest breakdown:
split  is_inlayout_holdout
test   False                  2000
train  False                  5760
       True                    640
val    False                  1600
dtype: int64

--- flip_y=True | Tagged 62 / 81 words ---
[('Address:4894', 'B-SELLER_ADDRESS'), ('Melanie', 'I-SELLER_ADDRESS'), ('Pass', 'I-SELLER_ADDRESS'), ('Apt.', 'I-SELLER_ADDRESS'), ('742', 'I-SELLER_ADDRESS'), ('Tammyland,', 'I-SELLER_ADDRESS'), ('SD', 'I-SELLER_ADDRESS'), ('42587', 'I-SELLER_ADDRESS'), ('US', 'I-SELLER_ADDRESS'), ('Date:', 'B-DATE'), ('03-Jan-1994', 'I-DATE'), ('INVOICE', 'B-NUMBER'), ('#', 'I-NUMBER'), ('INV/30-14/832', 'I-NUMBER'), ('Due', 'B-DUE_DATE'), ('Date', 'I-DUE_DATE'), (':', 'I-DUE_DATE'), ('06-Feb-2007', 'I-DUE_DATE'), ('Terms', 'B-CONDITIONS'), ('and', 'I-CONDITIONS')]

--- flip_y=False | Tagged 0 / 81 words ---
[]


In [ ]:
import json
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

def build_split_records(sub_df):
    records = []
    zero_tag_docs = 0
    for row in tqdm(sub_df.itertuples(index=False), total=len(sub_df)):
        ocr_rec = ocr_index.get(row.img_path)
        if not ocr_rec or not ocr_rec["words"]:
            continue

        W, H = ocr_rec["width"], ocr_rec["height"]
        words = [w[0] for w in ocr_rec["words"]]
        bboxes = [
            [
                max(0, min(1000, int(round(w[2] * 1000)))),
                max(0, min(1000, int(round(w[3] * 1000)))),
                max(0, min(1000, int(round(w[4] * 1000)))),
                max(0, min(1000, int(round(w[5] * 1000)))),
            ]
            for w in ocr_rec["words"]
        ]

        raw_ann = json.loads((ANN_DIR / row.annot_path).read_text(encoding="utf-8"))
        norm_ann = scale_ann_to_1000(cannon(raw_ann), W, H, flip_y=True)
        tags = label_one_invoice(words, bboxes, norm_ann)

        if all(t == "O" for t in tags):
            zero_tag_docs += 1

        records.append({
            "doc_id": row.img_path,
            "template_id": int(row.template_id),
            "words": words,
            "bboxes": bboxes,
            "tags": tags,
        })
    return records, zero_tag_docs

train_df         = df[(df["split"] == "train") & (~df["is_inlayout_holdout"])]
val_df           = df[df["split"] == "val"]
test_inlayout_df = df[(df["split"] == "train") & (df["is_inlayout_holdout"])]
test_disjoint_df = df[df["split"] == "test"]

# Leakage assertion: train aur val/test_disjoint mein template overlap ZERO hona chahiye
assert set(train_df.template_id).isdisjoint(set(val_df.template_id))
assert set(train_df.template_id).isdisjoint(set(test_disjoint_df.template_id))
assert set(val_df.template_id).isdisjoint(set(test_disjoint_df.template_id))

train_data, z_tr = build_split_records(train_df)
val_data, z_va   = build_split_records(val_df)
test_in_data, _  = build_split_records(test_inlayout_df)
test_dis_data, _ = build_split_records(test_disjoint_df)

# Label mapping auto-build (O hamesha index 0)
all_tags = sorted({t for rec in train_data + val_data for t in rec["tags"] if t != "O"})
label_list = ["O"] + all_tags
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

print(f"Train: {len(train_data)} (0-tag: {z_tr}) | Val: {len(val_data)} (0-tag: {z_va})")
print(f"Test In-Layout: {len(test_in_data)} | Test Layout-Disjoint: {len(test_dis_data)}")
print(f"Total BIO labels ({len(label_list)}):", label_list[:10], "...")

  0%|          | 0/5760 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/640 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

Train: 5760 (0-tag: 0) | Val: 1600 (0-tag: 0)
Test In-Layout: 640 | Test Layout-Disjoint: 2000
Total BIO labels (45): ['O', 'B-AMOUNT_DUE', 'B-CONDITIONS', 'B-CUSTOMER', 'B-DATE', 'B-DISCOUNT', 'B-DUE_DATE', 'B-GSTIN', 'B-GSTIN_BUYER', 'B-GSTIN_SELLER'] ...


In [ ]:
import inspect
import json
import math
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

# 1. Verify CUDA GPU availability before starting training
assert torch.cuda.is_available(), (
    "CUDA GPU not detected. Enable GPU under Kaggle Session Options -> Accelerator."
)
print(f"Active GPU: {torch.cuda.get_device_name(0)} | GPU Count: {torch.cuda.device_count()}")

# Configuration
MODEL_NAME = "SCUT-DLVCLab/lilt-roberta-en-base"
CKPT_DIR = Path("/kaggle/working/lilt_fatura_ckpt")
SAVE_DIR = Path("/kaggle/working/lilt_fatura_best")
HF_REPO_ID = "Satyamkarn100/findocai-lilt-fatura"

# 2. Initialize tokenizer (resolves to LayoutLMv3Tokenizer with native box/label alignment)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode_batch(batch: dict) -> dict:
    """
    Tokenize word lists with aligned 0-1000 bounding boxes and word-level BIO labels.
    Applies a 512-token sliding window with a 128-token stride for long documents.
    """
    word_label_ids = [
        [label2id.get(tag, 0) for tag in tags]
        for tags in batch["tags"]
    ]
    encoded = tokenizer(
        batch["words"],
        boxes=batch["bboxes"],
        word_labels=word_label_ids,
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        padding="max_length",
    )
    # Remove auxiliary tokenizer outputs not accepted by LiltForTokenClassification
    encoded.pop("overflow_to_sample_mapping", None)
    encoded.pop("offset_mapping", None)
    return encoded


def to_hf_dataset(records: list) -> Dataset:
    """Convert raw record dictionaries into a tokenized Hugging Face Dataset."""
    ds = Dataset.from_list(records)
    return ds.map(encode_batch, batched=True, remove_columns=ds.column_names)


# Build tokenized datasets from existing in-memory splits
train_ds = to_hf_dataset(train_data)
val_ds = to_hf_dataset(val_data)
test_in_ds = to_hf_dataset(test_in_data)
test_dis_ds = to_hf_dataset(test_dis_data)

print(f"Encoded chunks -> Train: {len(train_ds)} | Val: {len(val_ds)}")
print("Dataset columns:", train_ds.column_names)


# 3. Define entity-level evaluation metrics (ignoring subword/padding -100 tokens)
def compute_metrics(eval_pred) -> dict:
    logits, label_ids = eval_pred
    preds = np.argmax(logits, axis=2)

    true_preds = [
        [id2label[pr] for pr, lb in zip(pred_row, lab_row) if lb != -100]
        for pred_row, lab_row in zip(preds, label_ids)
    ]
    true_labs = [
        [id2label[lb] for pr, lb in zip(pred_row, lab_row) if lb != -100]
        for pred_row, lab_row in zip(preds, label_ids)
    ]

    return {
        "precision": precision_score(true_labs, true_preds, zero_division=0),
        "recall": recall_score(true_labs, true_preds, zero_division=0),
        "f1": f1_score(true_labs, true_preds, zero_division=0),
    }


# 4. Initialize LiLT token classification model and version-safe TrainingArguments
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

NUM_EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM = 2
num_gpus = max(1, torch.cuda.device_count())
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM * num_gpus))
warmup_steps = int(0.1 * steps_per_epoch * NUM_EPOCHS)

ta_params = inspect.signature(TrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

raw_args = {
    "output_dir": str(CKPT_DIR),
    eval_key: "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": 0.01,
    "warmup_steps": warmup_steps,
    "fp16": True,
    "logging_steps": 50,
    "save_total_limit": 2,
    "report_to": "none",
}

# Filter arguments dynamically for compatibility across transformers versions
safe_args = {k: v for k, v in raw_args.items() if k in ta_params}
training_args = TrainingArguments(**safe_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

# 5. Train model and persist the best checkpoint
trainer.train()

SAVE_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print(f"Best LiLT model and tokenizer saved to: {SAVE_DIR}")

# 6. Evaluate generalization on in-layout vs. layout-disjoint test splits
eval_summary = {}
report_sections = []

for split_name, dataset in [
    ("test_inlayout_640", test_in_ds),
    ("test_disjoint_2000", test_dis_ds),
]:
    output = trainer.predict(dataset)
    preds = np.argmax(output.predictions, axis=2)

    true_preds = [
        [id2label[pr] for pr, lb in zip(pred_row, lab_row) if lb != -100]
        for pred_row, lab_row in zip(preds, output.label_ids)
    ]
    true_labs = [
        [id2label[lb] for pr, lb in zip(pred_row, lab_row) if lb != -100]
        for pred_row, lab_row in zip(preds, output.label_ids)
    ]

    metrics = {
        "precision": round(float(precision_score(true_labs, true_preds, zero_division=0)), 4),
        "recall": round(float(recall_score(true_labs, true_preds, zero_division=0)), 4),
        "f1": round(float(f1_score(true_labs, true_preds, zero_division=0)), 4),
    }
    report = classification_report(true_labs, true_preds, digits=4, zero_division=0)

    eval_summary[split_name] = metrics
    section = f"================ {split_name} ================\nMetrics: {metrics}\n\n{report}\n"
    report_sections.append(section)
    print("\n" + section)

# Save evaluation artifacts alongside model weights
(SAVE_DIR / "eval_metrics.json").write_text(
    json.dumps(eval_summary, indent=2), encoding="utf-8"
)
(SAVE_DIR / "classification_reports.txt").write_text(
    "\n".join(report_sections), encoding="utf-8"
)

# 7. Upload model weights, tokenizer, and evaluation reports to Hugging Face Hub
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=True,
    exist_ok=True,
)

api.upload_folder(
    folder_path=str(SAVE_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload fine-tuned LiLT-RoBERTa on FATURA (layout-disjoint split) with evaluation metrics",
)

print(f"Upload complete -> https://huggingface.co/{HF_REPO_ID}")

Active GPU: Tesla T4 | GPU Count: 2


Map:   0%|          | 0/5760 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/640 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Encoded chunks -> Train: 5760 | Val: 1600
Dataset columns: ['input_ids', 'attention_mask', 'bbox', 'labels']


... (14 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



================ test_inlayout_640 ================
Metrics: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

                 precision    recall  f1-score   support

     AMOUNT_DUE     1.0000    1.0000    1.0000        60
     CONDITIONS     1.0000    1.0000    1.0000       191
       CUSTOMER     1.0000    1.0000    1.0000       610
           DATE     1.0000    1.0000    1.0000       620
       DISCOUNT     1.0000    1.0000    1.0000       160
       DUE_DATE     1.0000    1.0000    1.0000       380
          GSTIN     1.0000    1.0000    1.0000        80
    GSTIN_BUYER     1.0000    1.0000    1.0000        40
   GSTIN_SELLER     1.0000    1.0000    1.0000        40
           NOTE     1.0000    1.0000    1.0000       437
         NUMBER     1.0000    1.0000    1.0000       560
PAYMENT_DETAILS     1.0000    1.0000    1.0000       224
      PO_NUMBER     1.0000    1.0000    1.0000       100
 SELLER_ADDRESS     1.0000    1.0000    1.0000       546
   SELLER_EMAIL     1.0000    1.0000


================ test_disjoint_2000 ================
Metrics: {'precision': 0.9375, 'recall': 0.9471, 'f1': 0.9422}

                 precision    recall  f1-score   support

     AMOUNT_DUE     0.9950    1.0000    0.9975       199
     CONDITIONS     1.0000    1.0000    1.0000       481
       CUSTOMER     0.8873    0.8894    0.8883      2460
           DATE     1.0000    1.0000    1.0000      2000
       DISCOUNT     0.6250    0.6250    0.6250       400
       DUE_DATE     1.0000    1.0000    1.0000       800
          GSTIN     0.5000    0.6667    0.5714       600
   GSTIN_SELLER     0.0000    0.0000    0.0000       400
           NOTE     1.0000    1.0000    1.0000       656
         NUMBER     1.0000    1.0000    1.0000      2000
PAYMENT_DETAILS     1.0000    1.0000    1.0000       288
      PO_NUMBER     1.0000    1.0000    1.0000       200
 SELLER_ADDRESS     0.9994    0.9994    0.9994      1708
   SELLER_EMAIL     1.0000    1.0000    1.0000      1200
    SELLER_NAME     0.9685

BackendError: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 145996430 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.

In [ ]:
from getpass import getpass
from pathlib import Path
from huggingface_hub import HfApi

SAVE_DIR = Path("/kaggle/working/lilt_fatura_best")
HF_REPO_ID = "Satyamkarn100/findocai-lilt-fatura"

assert SAVE_DIR.exists(), f"Missing directory: {SAVE_DIR}"
print("Files ready to upload:", [p.name for p in sorted(SAVE_DIR.iterdir())])

# Try Kaggle Secret first; fall back to secure prompt if not attached to this notebook
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = getpass("Paste your Hugging Face WRITE token: ").strip()

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=True,
    exist_ok=True,
)

api.upload_folder(
    folder_path=str(SAVE_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Upload fine-tuned LiLT-RoBERTa on FATURA (layout-disjoint F1: 0.9422)",
)

print(f"Upload complete -> https://huggingface.co/{HF_REPO_ID}")

Files ready to upload: ['classification_reports.txt', 'config.json', 'eval_metrics.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
Upload complete -> https://huggingface.co/Satyamkarn100/findocai-lilt-fatura


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
print("Files on HF Hub:", api.list_repo_files("Satyamkarn100/findocai-lilt-fatura", repo_type="model"))

Files on HF Hub: ['.gitattributes', 'classification_reports.txt', 'config.json', 'eval_metrics.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
